***Unified Military Analytics and Comparison Project***

## Objective
Automate the collection, cleaning, and analysis of global military power data for 140+ countries and engineer analytical KPIs for dashboard consumption.



## Data Source
***GlobalFirepower.com (Open-source military statistics)***

In [ ]:
##Importing necessary Libraries
import pandas as pd
import numpy as np
import requests
import re
import time
from bs4 import BeautifulSoup

**GET COUNTRY IDS**

In [ ]:
headers = {"User-Agent": "Mozilla/5.0"}

countries_url = "https://www.globalfirepower.com/countries-listing.php"
response = requests.get(countries_url, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

ids = []
for a in soup.find_all("a", href=True):
    if "country-military-strength-detail.php?country_id=" in a["href"]:
        ids.append(a["href"].split("country_id=")[1])

ids = list(set(ids))
print("Total Countries Found:", len(ids))

Total Countries Found: 145


**FULL METRIC MAPPING**
## Task 3: Metric Mapping (key_map)

A key-value mapping is created where:
- Key → Column name in dataset
- Value → Label used on the website

This allows flexible scraping without hardcoding HTML positions.

In [ ]:
key_map = {
    # Population & Economy
    "total_population": "Total Population:",
    "gdp": "Purchasing Power Parity:",
    "defense_budget": "Defense Budget:",
    "external_debt": "External Debt:",

    # Manpower
    "total_manpower": "Available Manpower",
    "active_personnel": "Active Personnel",
    "reserve_personnel": "Reserve Personnel",
    "air_force_personnel": "Air Force Personnel*",
    "army_personnel": "Army Personnel*",
    "navy_personnel": "Navy Personnel*",

    # Air Power
    "total_aircraft": "Aircraft Total:",
    "attack_aircraft": "Attack Types:",
    "fighter_aircraft": "Fighters:",
    "transport_aircraft": "Transports (Fixed-Wing):",
    "helicopters": "Helicopters:",
    "trainer_aircraft": "Trainers:",
    "special_mission_aircraft": "Special-Mission:",
    "attack_helicopters": "Attack Helicopters:",
    "tanker_aircraft": "Tanker Fleet:",

    # Naval Power
    "naval_assets": "Total Assets:",
    "aircraft_carriers": "Aircraft Carriers:",
    "helicopter_carriers": "Helicopter Carriers:",
    "destroyers": "Destroyers:",
    "frigates": "Frigates:",
    "corvettes": "Corvettes:",
    "submarines": "Submarines:",
    "patrol_vessel": "Patrol Vessels:",
    "mine_warfare": "Mine Warfare:",

    # Ground Forces
    "tanks": "Tanks:",
    "self_propelled_artillery": "Self-Propelled Artillery:",
    "towed_artillery": "Towed Artillery:",
    "rocket_artillery": "MLRS (Rocket Artillery):"
}

**SCRAPING LOGIC**

**Automated Web Scraping**

**This function:**
- Iterates through all country URLs
- Extracts power index and all mapped metrics
- Stores values exactly as found (raw text)
- Returns a structured DataFrame

In [ ]:
def scrape_all_countries(ids, key_map):
    base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="
    all_data = {}

    for cid in ids:
        try:
            url = base_url + cid
            soup = BeautifulSoup(
                requests.get(url, headers=headers, timeout=10).text,
                "html.parser"
            )

            country_name = cid.replace("-", " ").title()

            # Power Index
            power_index = None
            pi_tag = soup.select_one("span.textNormal.textDkGray")
            if pi_tag:
                match = re.search(r"\d+\.\d+", pi_tag.text)
                if match:
                    power_index = match.group()

            # Extract specs
            data_found = {}
            for block in soup.find_all("div", class_="specsGenContainers"):
                label = block.select_one("span.textLarge.textBold")
                values = block.select("span.textWhite")
                if label and values:
                    data_found[label.text.strip()] = values[-1].text.strip()

            row = {k: data_found.get(v) for k, v in key_map.items()}
            row["power_index"] = power_index

            all_data[country_name] = row
            print("Scraped:", country_name)
            time.sleep(1)

        except Exception as e:
            print("Error:", cid, e)

    return pd.DataFrame.from_dict(all_data, orient="index").reset_index().rename(columns={"index": "country"})

**EXPORT RAW DATASET**

In [ ]:
raw_df = scrape_all_countries(ids, key_map)
raw_df.to_csv("military_raw_data.csv", index=False)
raw_df.head()

Scraped: Sri Lanka
Scraped: Croatia
Scraped: Suriname
Scraped: Netherlands
Scraped: Czech Republic
Scraped: Ethiopia
Scraped: South Sudan
Scraped: Spain
Scraped: Nigeria
Scraped: Mauritania
Scraped: Bangladesh
Scraped: Beliz
Scraped: Bahrain
Scraped: Kyrgyzstan
Scraped: Libya
Scraped: Tanzania
Scraped: Sudan
Scraped: Bolivia
Scraped: United States Of America
Scraped: Nepal
Scraped: Ghana
Scraped: Pakistan
Scraped: Mozambique
Scraped: Portugal
Scraped: Somalia
Scraped: Guatemala
Scraped: Paraguay
Scraped: Turkmenistan
Scraped: Panama
Scraped: Botswana
Scraped: Nicaragua
Scraped: Madagascar
Scraped: Cambodia
Scraped: Azerbaijan
Scraped: Slovenia
Scraped: Finland
Scraped: Namibia
Scraped: Singapore
Scraped: Austria
Scraped: Philippines
Scraped: Honduras
Scraped: Bhutan
Scraped: Democratic Republic Of The Congo
Scraped: Australia
Scraped: Israel
Scraped: Kuwait
Scraped: Iraq
Scraped: South Korea
Scraped: Lithuania
Scraped: Mexico
Scraped: Estonia
Scraped: Iceland
Scraped: Serbia
Scraped: L

,country,total_population,gdp,defense_budget,external_debt,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,...,frigates,corvettes,submarines,patrol_vessel,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,power_index
0,Sri Lanka,"21,982,608","$301,407,000,000 USD","$1,500,000,000 USD","$42,198,000,000 USD","10,551,652","346,000","90,000","26,000","150,000",...,5,0,0,259,0,Stock: 87\t\t\t\t\nReadiness: 52*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 96\t\t\t\t\nReadiness: 58*,Stock: 28\t\t\t\t\nReadiness: 17*,1.3822
1,Croatia,"4,150,116","$164,825,000,000 USD","$2,744,000,000 USD","$60,500,000,000 USD","2,033,557","14,325","20,100","1,500","7,000",...,0,0,0,11,1,Stock: 55\t\t\t\t\nReadiness: 44*,Stock: 25\t\t\t\t\nReadiness: 20*,Stock: 42\t\t\t\t\nReadiness: 34*,Stock: 42\t\t\t\t\nReadiness: 34*,1.5432
2,Suriname,"646,758","$12,316,000,000 USD","$58,440,000 USD","$2,645,000,000 USD","142,287","2,500",0,150,"1,000",...,0,0,0,17,0,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,4.0538
3,Netherlands,"17,772,378","$1,276,000,000,000 USD","$31,791,150,000 USD","$5,257,949,730,000 USD","7,997,570","44,245","8,510","9,500","53,212",...,6,0,3,76,3,Stock: 18\t\t\t\t\nReadiness: 13*,Stock: 46\t\t\t\t\nReadiness: 32*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 4\t\t\t\t\nReadiness: 3*,0.6142
4,Czech Republic,"10,837,890","$521,928,000,000 USD","$9,174,000,000 USD","$232,320,000,000 USD","5,093,808","30,334","4,900","3,120","13,000",...,None,None,None,None,None,Stock: 20\t\t\t\t\nReadiness: 14*,Stock: 53\t\t\t\t\nReadiness: 37*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 16\t\t\t\t\nReadiness: 11*,1.0313


**DATA CLEANING**

## Task 6: Data Cleaning & Standardization

### Purpose of Data Cleaning

The raw data scraped from GlobalFirepower.com contains values stored as text.
These values include:
- Commas (e.g., "1,200,000")
- Currency symbols (e.g., "$50,000,000")
- Percentage symbols (e.g., "3.2%")
- Extra text labels (e.g., "Stock:", "Approx.")
- Missing values such as "N/A" or blank fields

Such text-based values cannot be used directly for mathematical calculations.
Therefore, data cleaning is required to convert all relevant columns into
pure numeric formats while preserving data meaning.

### Objectives of Cleaning
- Convert all numeric-looking text into numbers
- Standardize column names for consistency
- Handle missing values logically
- Prepare the dataset for KPI engineering and analytics

The output of this task is a clean, numeric, analytics-ready dataset.

**STANDARDIZE COLUMN NAMES**

In [ ]:
df = pd.read_csv("military_raw_data.csv")
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

print("Standardized Columns:")
df.columns.tolist()

Standardized Columns:


['country',
 'total_population',
 'gdp',
 'defense_budget',
 'external_debt',
 'total_manpower',
 'active_personnel',
 'reserve_personnel',
 'air_force_personnel',
 'army_personnel',
 'navy_personnel',
 'total_aircraft',
 'attack_aircraft',
 'fighter_aircraft',
 'transport_aircraft',
 'helicopters',
 'trainer_aircraft',
 'special_mission_aircraft',
 'attack_helicopters',
 'tanker_aircraft',
 'naval_assets',
 'aircraft_carriers',
 'helicopter_carriers',
 'destroyers',
 'frigates',
 'corvettes',
 'submarines',
 'patrol_vessel',
 'mine_warfare',
 'tanks',
 'self_propelled_artillery',
 'towed_artillery',
 'rocket_artillery',
 'power_index']

**IDENTIFY DATA NOISE**

In [ ]:
df.describe(include="all")

,country,total_population,gdp,defense_budget,external_debt,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,...,frigates,corvettes,submarines,patrol_vessel,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,power_index
count,145,145,145,145,145,145,145,145,145,145,...,113.000000,113.000000,113.000000,113.000000,113.000000,145,145,145,145,145.000000
unique,145,145,145,144,141,145,126,85,110,129,...,NaN,NaN,NaN,NaN,NaN,109,91,111,93,NaN
top,Sri Lanka,"21,982,608","$301,407,000,000 USD","$5,100,000,000 USD","$60,500,000,000 USD","10,551,652","25,000",0,0,"130,000",...,NaN,NaN,NaN,NaN,NaN,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,NaN
freq,1,1,1,2,3,1,5,52,6,3,...,NaN,NaN,NaN,NaN,NaN,37,46,34,45,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.663717,3.548673,4.230088,42.442478,3.637168,NaN,NaN,NaN,NaN,1.642772
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.294287,9.720184,11.218614,54.450656,7.150252,NaN,NaN,NaN,NaN,1.135182
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.074100
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,8.000000,0.000000,NaN,NaN,NaN,NaN,0.651700
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,20.000000,0.000000,NaN,NaN,NaN,NaN,1.543200
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.000000,3.000000,4.000000,55.000000,4.000000,NaN,NaN,NaN,NaN,2.396900


**CLEAN NUMERIC VALUES**

In [ ]:
def clean_numeric_value(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # Remove common noise
    x = (
        x.replace(",", "")
         .replace("%", "")
         .replace("+", "")
         .replace("$", "")
    )

    # Handle text-based nulls
    if x.lower() in ["n/a", "na", "null", "", "none"]:
        return np.nan

    # Extract numeric part using regex
    match = re.search(r"[\d.]+", x)
    return float(match.group()) if match else np.nan

**APPLY CLEANING TO ALL METRIC COLUMNS**

In [ ]:
numeric_columns = [col for col in df.columns if col != "country"]

for col in numeric_columns:
    df[col] = df[col].apply(clean_numeric_value)

df.head()

,country,total_population,gdp,defense_budget,external_debt,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,...,frigates,corvettes,submarines,patrol_vessel,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,power_index
0,Sri Lanka,21982608.0,3.014070e+11,1.500000e+09,4.219800e+10,10551652.0,346000.0,90000.0,26000.0,150000.0,...,5.0,0.0,0.0,259.0,0.0,87.0,0.0,96.0,28.0,1.3822
1,Croatia,4150116.0,1.648250e+11,2.744000e+09,6.050000e+10,2033557.0,14325.0,20100.0,1500.0,7000.0,...,0.0,0.0,0.0,11.0,1.0,55.0,25.0,42.0,42.0,1.5432
2,Suriname,646758.0,1.231600e+10,5.844000e+07,2.645000e+09,142287.0,2500.0,0.0,150.0,1000.0,...,0.0,0.0,0.0,17.0,0.0,0.0,0.0,0.0,0.0,4.0538
3,Netherlands,17772378.0,1.276000e+12,3.179115e+10,5.257950e+12,7997570.0,44245.0,8510.0,9500.0,53212.0,...,6.0,0.0,3.0,76.0,3.0,18.0,46.0,0.0,4.0,0.6142
4,Czech Republic,10837890.0,5.219280e+11,9.174000e+09,2.323200e+11,5093808.0,30334.0,4900.0,3120.0,13000.0,...,NaN,NaN,NaN,NaN,NaN,20.0,53.0,0.0,16.0,1.0313


**HANDLE MISSING VALUES**

- Missing values occur mainly for naval assets in landlocked countries
- These represent absence of assets, not data errors
- Therefore, missing values are filled with 0

In [ ]:
missing_summary = df.isnull().sum()
missing_summary[missing_summary > 0]
df.fillna(0, inplace=True)

## 🔍 Data Quality Assessment

Before performing analysis, the dataset was evaluated for:

- Missing values
- Data types
- Consistency across columns

### Observations:
- Most columns are successfully converted into numeric format
- Missing values are minimal and handled appropriately
- No major structural inconsistencies found

This ensures that the dataset is reliable for further analysis and visualization.

**VALIDATE CLEANED DATA**

In [ ]:
# Row count validation
print("Row Count:", len(df))

# Check for remaining nulls
print("Total Null Values:", df.isnull().sum().sum())

# Check data types
df.dtypes

Row Count: 145
Total Null Values: 0


,0
country,object
total_population,float64
gdp,float64
defense_budget,float64
external_debt,float64
total_manpower,float64
active_personnel,float64
reserve_personnel,float64
air_force_personnel,float64
army_personnel,float64


**EXPORT CLEANED DATASET**

In [ ]:
df.to_csv("military_cleaned.csv", index=False)
print("military_cleaned.csv successfully created")

military_cleaned.csv successfully created


##  KPI Engineering

To enhance the analytical capability of the dataset, several Key Performance Indicators (KPIs) were created:

### 1️⃣ Assets per Capita
Measures the availability of military assets relative to population size.

This helps identify countries that are heavily militarized relative to their population.

### 2️⃣ Budget-to-GDP Ratio
Represents the proportion of national economic output allocated to defense.

This indicates how much priority a country gives to military spending.

### 3️⃣ Power Index Rank Gap
Measures the difference between a country’s rank and the global average rank.

This helps identify countries that significantly outperform or underperform relative to others.

These KPIs enable deeper comparative insights beyond raw metrics.

**KPI CALCULATIONS**

In [ ]:
df = pd.read_csv("military_cleaned.csv")

df["total_military_assets"] = (
    df["total_aircraft"] +
    df["tanks"] +
    df["naval_assets"]
)

df["assets_per_capita"] = df["total_military_assets"] / df["total_manpower"].replace(0, 1)

df["budget_to_gdp_ratio"] = df["defense_budget"] / df["gdp"].replace(0, 1)

df["rank"] = df["power_index"].rank(ascending=True, method="min").astype(int)
df["power_index_rank_gap"] = df["rank"] - df["rank"].min()

df.to_csv("military_final.csv", index=False)